In [1]:
# ============================================================
# Cell 0 | 项目根目录定位 + 10点留出实验输出目录
# 规则：
# 1) 原项目 data 仅用于读取未变化的其他性质数据
# 2) 本次实验所有新输出均保存到 small_paper_data
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for p in [start] + list(start.parents):
        if (
            (p / "data").exists()
            and (p / "notebooks").exists()
        ):
            return p

    raise FileNotFoundError(
        "未找到同时包含 data 和 notebooks 文件夹的项目根目录，"
        "请检查项目结构。"
    )


PROJ_ROOT = find_project_root(Path.cwd())

# 原项目数据目录：只读取未发生变化的其他性质数据
DATA_DIR = PROJ_ROOT / "data"

# 本次10点留出验证的独立实验目录
EXPERIMENT_ROOT = (
    PROJ_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

# 本Notebook所有输出保存到这里
OUT_DIR = EXPERIMENT_ROOT / "ml_ready"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if "small_paper_data" not in str(OUT_DIR):
    raise RuntimeError(
        "输出目录设置异常，已停止运行，防止覆盖原项目文件。"
    )

print("PROJ_ROOT       =", PROJ_ROOT)
print("DATA_DIR        =", DATA_DIR)
print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)
print("OUT_DIR         =", OUT_DIR)

PROJ_ROOT       = D:\jupyter\IL_ML_Project
DATA_DIR        = D:\jupyter\IL_ML_Project\data
EXPERIMENT_ROOT = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
OUT_DIR         = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready


In [2]:
# ============================================================
# Cell 1 | 读取10点留出实验的黏度预处理结果
# ============================================================

VIS_PATH = (
    EXPERIMENT_ROOT
    / "data"
    / "viscosity"
    / "interim"
    / "vis_ml_ready.csv"
)

if not VIS_PATH.exists():
    raise FileNotFoundError(
        f"找不到黏度预处理结果：\n{VIS_PATH}\n"
        "请先完成01_粘度预处理_10点留出版.ipynb。"
    )

df_vis = pd.read_csv(VIS_PATH)

df_vis.columns = (
    df_vis.columns
    .astype(str)
    .str.strip()
)

required_columns = [
    "cation_smiles",
    "anion_smiles",
    "IL_key",
    "T_group",
    "P_group",
    "T_rep",
    "P_rep",
    "vis_rep_mpas",
    "vis_sigma_rep_mpas",
    "lg_eta",
    "lg_eta_sigma",
    "sample_weight_from_sigma",
    "n_points",
    "agg_method",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_vis.columns
]

if missing_columns:
    raise ValueError(
        "黏度预处理结果缺少必要字段：\n"
        f"{missing_columns}"
    )

print("读取文件：")
print(VIS_PATH)

print("\n数据形状：")
print(df_vis.shape)

print("\n字段名称：")
print(df_vis.columns.tolist())

print("\n关键字段缺失数量：")
display(
    df_vis[
        [
            "cation_smiles",
            "anion_smiles",
            "IL_key",
            "T_rep",
            "P_rep",
            "vis_rep_mpas",
            "lg_eta",
        ]
    ]
    .isna()
    .sum()
    .rename("缺失数量")
    .to_frame()
)

print("\n数据前5行：")
display(df_vis.head())

读取文件：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim\vis_ml_ready.csv

数据形状：
(16216, 14)

字段名称：
['cation_smiles', 'anion_smiles', 'IL_key', 'T_group', 'P_group', 'T_rep', 'P_rep', 'vis_rep_mpas', 'vis_sigma_rep_mpas', 'lg_eta', 'lg_eta_sigma', 'sample_weight_from_sigma', 'n_points', 'agg_method']

关键字段缺失数量：


,缺失数量
cation_smiles,0
anion_smiles,0
IL_key,0
T_rep,0
P_rep,0
vis_rep_mpas,0
lg_eta,0



数据前5行：


,cation_smiles,anion_smiles,IL_key,T_group,P_group,T_rep,P_rep,vis_rep_mpas,vis_sigma_rep_mpas,lg_eta,lg_eta_sigma,sample_weight_from_sigma,n_points,agg_method
0,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,303.0,1.0,303.0,1.01325,62.60,2.60,1.796574,0.018038,3073.500923,1,weighted_by_1_over_sigma2
1,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,313.0,1.0,313.0,1.01325,40.40,1.70,1.606381,0.018275,2994.306581,1,weighted_by_1_over_sigma2
2,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,323.0,1.0,323.0,1.01325,28.00,1.20,1.447158,0.018613,2886.588971,1,weighted_by_1_over_sigma2
3,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,333.0,1.0,333.0,1.01325,20.47,0.86,1.311118,0.018246,3003.794102,1,weighted_by_1_over_sigma2
4,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,343.0,1.0,343.0,1.01325,15.47,0.65,1.189490,0.018248,3003.207166,1,weighted_by_1_over_sigma2


In [3]:
# ============================================================
# Cell 2 | 构建只包含黏度的建模数据表
# ============================================================

# 保留原黏度预处理结果，避免直接修改df_vis
df_ml = df_vis.copy()

# 与原建模流程保持一致：
# vis_rep_mpas统一命名为eta
# vis_sigma_rep_mpas统一命名为eta_sigma
# n_points统一命名为n_eta
df_ml = df_ml.rename(
    columns={
        "vis_rep_mpas": "eta",
        "vis_sigma_rep_mpas": "eta_sigma",
        "n_points": "n_eta",
    }
)

# 黏度数据可用性标记
df_ml["has_eta"] = (
    df_ml["eta"].notna()
    & df_ml["lg_eta"].notna()
)

# 小论文只保留黏度建模所需字段
final_columns = [
    "cation_smiles",
    "anion_smiles",
    "IL_key",
    "T_group",
    "P_group",
    "T_rep",
    "P_rep",
    "eta",
    "eta_sigma",
    "lg_eta",
    "lg_eta_sigma",
    "sample_weight_from_sigma",
    "n_eta",
    "agg_method",
    "has_eta",
]

missing_final_columns = [
    column
    for column in final_columns
    if column not in df_ml.columns
]

if missing_final_columns:
    raise ValueError(
        "构建黏度建模数据时缺少字段：\n"
        f"{missing_final_columns}"
    )

df_ml = (
    df_ml[final_columns]
    .sort_values(
        by=[
            "cation_smiles",
            "anion_smiles",
            "T_rep",
            "P_rep",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("仅黏度建模数据形状：")
print(df_ml.shape)

print("\n字段名称：")
print(df_ml.columns.tolist())

print("\n关键字段缺失数量：")
display(
    df_ml[
        [
            "cation_smiles",
            "anion_smiles",
            "IL_key",
            "T_rep",
            "P_rep",
            "eta",
            "lg_eta",
            "n_eta",
            "has_eta",
        ]
    ]
    .isna()
    .sum()
    .rename("缺失数量")
    .to_frame()
)

print("\n黏度数据可用性：")
print(
    df_ml["has_eta"]
    .value_counts(dropna=False)
)

print("\n数据前5行：")
display(df_ml.head())

仅黏度建模数据形状：
(16216, 15)

字段名称：
['cation_smiles', 'anion_smiles', 'IL_key', 'T_group', 'P_group', 'T_rep', 'P_rep', 'eta', 'eta_sigma', 'lg_eta', 'lg_eta_sigma', 'sample_weight_from_sigma', 'n_eta', 'agg_method', 'has_eta']

关键字段缺失数量：


,缺失数量
cation_smiles,0
anion_smiles,0
IL_key,0
T_rep,0
P_rep,0
eta,0
lg_eta,0
n_eta,0
has_eta,0



黏度数据可用性：
has_eta
True    16216
Name: count, dtype: int64

数据前5行：


,cation_smiles,anion_smiles,IL_key,T_group,P_group,T_rep,P_rep,eta,eta_sigma,lg_eta,lg_eta_sigma,sample_weight_from_sigma,n_eta,agg_method,has_eta
0,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,303.0,1.0,303.0,1.01325,62.60,2.60,1.796574,0.018038,3073.500923,1,weighted_by_1_over_sigma2,True
1,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,313.0,1.0,313.0,1.01325,40.40,1.70,1.606381,0.018275,2994.306581,1,weighted_by_1_over_sigma2,True
2,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,323.0,1.0,323.0,1.01325,28.00,1.20,1.447158,0.018613,2886.588971,1,weighted_by_1_over_sigma2,True
3,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,333.0,1.0,333.0,1.01325,20.47,0.86,1.311118,0.018246,3003.794102,1,weighted_by_1_over_sigma2,True
4,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,343.0,1.0,343.0,1.01325,15.47,0.65,1.189490,0.018248,3003.207166,1,weighted_by_1_over_sigma2,True


In [4]:
# ============================================================
# Cell 3 | 构建黏度可用样本索引
# ============================================================

eta_idx = np.flatnonzero(
    df_ml["eta"]
    .notna()
    .to_numpy()
)

lg_eta_idx = np.flatnonzero(
    df_ml["lg_eta"]
    .notna()
    .to_numpy()
)

eta_sigma_idx = np.flatnonzero(
    df_ml["eta_sigma"]
    .notna()
    .to_numpy()
)

lg_eta_sigma_idx = np.flatnonzero(
    df_ml["lg_eta_sigma"]
    .notna()
    .to_numpy()
)

sample_weight_idx = np.flatnonzero(
    df_ml["sample_weight_from_sigma"]
    .notna()
    .to_numpy()
)


print("黏度可用样本索引数量：")
print("eta_idx                 :", len(eta_idx))
print("lg_eta_idx              :", len(lg_eta_idx))
print("eta_sigma_idx           :", len(eta_sigma_idx))
print("lg_eta_sigma_idx        :", len(lg_eta_sigma_idx))
print("sample_weight_idx       :", len(sample_weight_idx))


# 检查eta与lg_eta的有效样本是否完全一致
eta_lg_same = np.array_equal(
    eta_idx,
    lg_eta_idx,
)

print("\neta_idx与lg_eta_idx是否完全一致：")
print(eta_lg_same)


if len(eta_idx) != len(df_ml):
    raise RuntimeError(
        "存在无法用于黏度建模的记录，请检查eta缺失情况。"
    )

if len(lg_eta_idx) != len(df_ml):
    raise RuntimeError(
        "存在无法用于lg_eta建模的记录，请检查lg_eta缺失情况。"
    )

if not eta_lg_same:
    raise RuntimeError(
        "eta_idx与lg_eta_idx不一致，请检查对数转换结果。"
    )


print("\n黏度可用样本索引检查通过。")

黏度可用样本索引数量：
eta_idx                 : 16216
lg_eta_idx              : 16216
eta_sigma_idx           : 15575
lg_eta_sigma_idx        : 15575
sample_weight_idx       : 15575

eta_idx与lg_eta_idx是否完全一致：
True

黏度可用样本索引检查通过。


In [5]:
# ============================================================
# Cell 4 | 生成黏度数据覆盖情况汇总
# ============================================================

n_total = len(df_ml)

coverage_summary = pd.DataFrame(
    {
        "subset_name": [
            "eta",
            "lg_eta",
            "eta_sigma",
            "lg_eta_sigma",
            "sample_weight_from_sigma",
        ],
        "n_samples": [
            len(eta_idx),
            len(lg_eta_idx),
            len(eta_sigma_idx),
            len(lg_eta_sigma_idx),
            len(sample_weight_idx),
        ],
    }
)

coverage_summary["coverage_rate"] = (
    coverage_summary["n_samples"]
    / n_total
)

dataset_summary = pd.DataFrame(
    {
        "检查项目": [
            "建模数据总行数",
            "唯一离子液体数量",
            "阳离子种类数量",
            "阴离子种类数量",
            "eta有效样本数",
            "lg_eta有效样本数",
            "具有有效不确定度的样本数",
        ],
        "数量": [
            n_total,
            df_ml["IL_key"].nunique(),
            df_ml["cation_smiles"].nunique(),
            df_ml["anion_smiles"].nunique(),
            len(eta_idx),
            len(lg_eta_idx),
            len(eta_sigma_idx),
        ],
    }
)

print("黏度数据覆盖情况：")
display(coverage_summary)

print("\n黏度建模数据汇总：")
display(dataset_summary)

print("\nlg_eta范围：")
print(
    float(df_ml["lg_eta"].min()),
    "～",
    float(df_ml["lg_eta"].max()),
)

print("\n实验黏度范围（mPa·s）：")
print(
    float(df_ml["eta"].min()),
    "～",
    float(df_ml["eta"].max()),
)

if len(eta_idx) != n_total:
    raise RuntimeError(
        "eta有效样本数与建模数据总行数不一致。"
    )

if len(lg_eta_idx) != n_total:
    raise RuntimeError(
        "lg_eta有效样本数与建模数据总行数不一致。"
    )

print("\n黏度建模数据覆盖情况检查通过。")

黏度数据覆盖情况：


,subset_name,n_samples,coverage_rate
0,eta,16216,1.000000
1,lg_eta,16216,1.000000
2,eta_sigma,15575,0.960471
3,lg_eta_sigma,15575,0.960471
4,sample_weight_from_sigma,15575,0.960471



黏度建模数据汇总：


,检查项目,数量
0,建模数据总行数,16216
1,唯一离子液体数量,1295
2,阳离子种类数量,583
3,阴离子种类数量,202
4,eta有效样本数,16216
5,lg_eta有效样本数,16216
6,具有有效不确定度的样本数,15575



lg_eta范围：
-1.0 ～ 12.30102999566398

实验黏度范围（mPa·s）：
0.1 ～ 2000000000000.0

黏度建模数据覆盖情况检查通过。


In [6]:
# ============================================================
# Cell 5 | 保存黏度建模数据、索引和运行记录
# ============================================================

from datetime import datetime


# ------------------------------------------------------------
# 1. 设置输出文件路径
# ------------------------------------------------------------
ML_READY_CSV = (
    OUT_DIR
    / "ml_ready_A.csv"
)

ML_READY_UNITS_CSV = (
    OUT_DIR
    / "ml_ready_A_with_units.csv"
)

AVAILABLE_INDICES_NPZ = (
    OUT_DIR
    / "ml_ready_A_available_indices.npz"
)

COVERAGE_SUMMARY_CSV = (
    OUT_DIR
    / "ml_ready_A_coverage_summary.csv"
)

DATASET_SUMMARY_CSV = (
    OUT_DIR
    / "ml_ready_A_dataset_summary.csv"
)

MANIFEST_JSON = (
    OUT_DIR
    / "ml_ready_A_manifest.json"
)


# ------------------------------------------------------------
# 2. 保存程序使用的标准建模数据
# ------------------------------------------------------------
df_ml.to_csv(
    ML_READY_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 3. 保存带单位表头的人工查看版本
# ------------------------------------------------------------
df_ml_with_units = df_ml.rename(
    columns={
        "T_group": "T_group [K]",
        "P_group": "P_group [bar]",
        "T_rep": "T_rep [K]",
        "P_rep": "P_rep [bar]",
        "eta": "eta [mPa·s]",
        "eta_sigma": "eta_sigma [mPa·s]",
        "lg_eta": "lg_eta [lg(mPa·s)]",
    }
)

df_ml_with_units.to_csv(
    ML_READY_UNITS_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 4. 保存黏度可用样本索引
# ------------------------------------------------------------
np.savez(
    AVAILABLE_INDICES_NPZ,
    eta_idx=eta_idx,
    lg_eta_idx=lg_eta_idx,
    eta_sigma_idx=eta_sigma_idx,
    lg_eta_sigma_idx=lg_eta_sigma_idx,
    sample_weight_idx=sample_weight_idx,
)


# ------------------------------------------------------------
# 5. 保存汇总表
# ------------------------------------------------------------
coverage_summary.to_csv(
    COVERAGE_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)

dataset_summary.to_csv(
    DATASET_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 6. 保存运行记录
# ------------------------------------------------------------
manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "project_root": str(PROJ_ROOT),
    "experiment_root": str(EXPERIMENT_ROOT),
    "input_file": str(VIS_PATH),
    "output_directory": str(OUT_DIR),
    "output_files": {
        "ml_ready_A": str(ML_READY_CSV),
        "ml_ready_A_with_units": str(
            ML_READY_UNITS_CSV
        ),
        "available_indices": str(
            AVAILABLE_INDICES_NPZ
        ),
        "coverage_summary": str(
            COVERAGE_SUMMARY_CSV
        ),
        "dataset_summary": str(
            DATASET_SUMMARY_CSV
        ),
        "manifest": str(MANIFEST_JSON),
    },
    "target_property": "viscosity",
    "target_for_modeling": "lg_eta",
    "n_rows": int(len(df_ml)),
    "n_unique_ionic_liquids": int(
        df_ml["IL_key"].nunique()
    ),
    "n_unique_cations": int(
        df_ml["cation_smiles"].nunique()
    ),
    "n_unique_anions": int(
        df_ml["anion_smiles"].nunique()
    ),
    "n_eta": int(len(eta_idx)),
    "n_lg_eta": int(len(lg_eta_idx)),
    "n_with_valid_sigma": int(
        len(eta_sigma_idx)
    ),
    "eta_min_mPa_s": float(
        df_ml["eta"].min()
    ),
    "eta_max_mPa_s": float(
        df_ml["eta"].max()
    ),
    "lg_eta_min": float(
        df_ml["lg_eta"].min()
    ),
    "lg_eta_max": float(
        df_ml["lg_eta"].max()
    ),
    "columns": df_ml.columns.tolist(),
}

with open(
    MANIFEST_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 7. 输出保存结果
# ------------------------------------------------------------
print("保存完成：")

print("\n黏度建模数据：")
print(ML_READY_CSV)

print("\n带单位表头的数据：")
print(ML_READY_UNITS_CSV)

print("\n可用样本索引：")
print(AVAILABLE_INDICES_NPZ)

print("\n覆盖情况汇总：")
print(COVERAGE_SUMMARY_CSV)

print("\n数据集数量汇总：")
print(DATASET_SUMMARY_CSV)

print("\n运行记录：")
print(MANIFEST_JSON)

保存完成：

黏度建模数据：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A.csv

带单位表头的数据：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A_with_units.csv

可用样本索引：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A_available_indices.npz

覆盖情况汇总：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A_coverage_summary.csv

数据集数量汇总：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A_dataset_summary.csv

运行记录：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A_manifest.json
